In [2]:
import torch
from transformers.models.longformer.modeling_longformer import LongformerSelfAttention
from transformers.models.longformer.configuration_longformer import LongformerConfig


device = "cuda"
batch = 2
seq_len = 4096
hidden = 128
heads = 8
window = 128
warmup = 10
iters = 50

config = LongformerConfig(
    hidden_size=hidden,
    num_attention_heads=heads,
    attention_window=[window],
)

attn = LongformerSelfAttention(config, layer_id=0).to(device).eval()

x = torch.randn(batch, seq_len, hidden, device=device)

attention_mask = torch.zeros(batch, seq_len, device=device)
is_index_masked = torch.zeros(batch, seq_len, dtype=torch.bool, device=device)
is_index_global_attn = torch.zeros(batch, seq_len, dtype=torch.bool, device=device)

# Warmup
with torch.no_grad():
    for _ in range(warmup):
        attn(
            x,
            attention_mask=attention_mask,
            layer_head_mask=None,
            is_index_masked=is_index_masked,
            is_index_global_attn=is_index_global_attn,
            is_global_attn=False,
        )

torch.cuda.synchronize()

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

times = []

with torch.no_grad():
    for _ in range(iters):
        start.record()
        attn(
            x,
            attention_mask=attention_mask,
            layer_head_mask=None,
            is_index_masked=is_index_masked,
            is_index_global_attn=is_index_global_attn,
            is_global_attn=False,
        )
        end.record()
        torch.cuda.synchronize()
        times.append(start.elapsed_time(end))

print(f"Longformer SWA avg latency: {sum(times)/len(times):.2f} ms")


Longformer SWA avg latency: 4.47 ms
